In [1]:
%%writefile suppliers.json
[
 {
  "supplier_id": "S201",
  "supplier_name": "TechSource India",
  "city": "Hyderabad",
  "rating": 4.5,
  "contact": {
   "phone": "9876500011",
   "email": "techsource@mail.com"
  }
 },
 {
  "supplier_id": "S202",
  "supplier_name": "MobileWorld Distributors",
  "city": "Bangalore",
  "rating": 4.2,
  "contact": {
   "phone": null,
   "email": "mobileworld@mail.com"
  }
 },
 {
  "supplier_id": "S203",
  "supplier_name": "HomeTech Supply",
  "city": "Mumbai",
  "rating": 4.4,
  "contact": {
   "phone": "9876500013",
   "email": null
  }
 },
 {
  "supplier_id": "S204",
  "supplier_name": "Urban Furniture Co",
  "city": "Delhi",
  "rating": 4.0,
  "contact": {
   "phone": "9876500014",
   "email": "urban@mail.com"
  }
 },
 {
  "supplier_id": "S205",
  "supplier_name": "Fashion Direct",
  "city": "Pune",
  "rating": 3.8,
  "contact": {
   "phone": null,
   "email": null
  }
 }
]

Writing suppliers.json


In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('join').getOrCreate()

In [4]:
from google.colab import files
uploaded = files.upload()

Saving inventory.csv to inventory.csv
Saving products.csv to products.csv
Saving sales.csv to sales.csv
Saving stores.csv to stores.csv


In [5]:
stores_df = spark.read.csv(
    "stores.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "products.csv",
    header=True,
    inferSchema=True
)

inventory_df = spark.read.csv(
    "inventory.csv",
    header=True,
    inferSchema=True
)

sales_df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

suppliers_df = spark.read.option(
    "multiline",
    "true"
).json(
    "suppliers.json"
)

In [6]:
#1-4
stores_df.show()
products_df.show()
inventory_df.show()
sales_df.show()

+--------+--------------------+---------+-----------+-----------+------------+
|store_id|          store_name|     city|      state| store_type|manager_name|
+--------+--------------------+---------+-----------+-----------+------------+
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya Reddy|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|Hypermarket|  Amit Kumar|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|Supermarket| Sneha Patel|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|Hypermarket|  Farhan Ali|
|    S106|     Metro Mart Pune|     Pune|Maharashtra| Mini Store|  Neha Singh|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala| Mini Store| Arjun Verma|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|Supermarket|  Meera Nair|
+--------+--------------------+---------+-----------+-----------+------------+

+----------+------------+-----------+------------+-

In [7]:
#5
suppliers_df.show(truncate=False)

+---------+---------------------------------+------+-----------+------------------------+
|city     |contact                          |rating|supplier_id|supplier_name           |
+---------+---------------------------------+------+-----------+------------------------+
|Hyderabad|{techsource@mail.com, 9876500011}|4.5   |S201       |TechSource India        |
|Bangalore|{mobileworld@mail.com, NULL}     |4.2   |S202       |MobileWorld Distributors|
|Mumbai   |{NULL, 9876500013}               |4.4   |S203       |HomeTech Supply         |
|Delhi    |{urban@mail.com, 9876500014}     |4.0   |S204       |Urban Furniture Co      |
|Pune     |{NULL, NULL}                     |3.8   |S205       |Fashion Direct          |
+---------+---------------------------------+------+-----------+------------------------+



In [11]:
#6
stores_df.printSchema()

products_df.printSchema()

inventory_df.printSchema()

sales_df.printSchema()

suppliers_df.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: int

In [10]:
#7
print("Stores:", stores_df.count())

print("Products:", products_df.count())

print("Inventory:", inventory_df.count())

print("Sales:", sales_df.count())

print("Suppliers:", suppliers_df.count())

Stores: 8
Products: 12
Inventory: 12
Sales: 15
Suppliers: 5


In [12]:
#8
stores_df.write.mode(
    "overwrite"
).parquet(
    "bronze_stores"
)

In [15]:
#9
products_df.write.mode(
    "overwrite"
).parquet(
    "bronze_stores"
)

In [16]:
#10
inventory_df.write.mode(
    "overwrite"
).parquet(
    "bronze_inventory"
)

sales_df.write.mode(
    "overwrite"
).parquet(
    "bronze_sales"
)

suppliers_df.write.mode(
    "overwrite"
).parquet(
    "bronze_suppliers"
)

In [17]:
#11
products_df.filter(
    products_df.supplier_id.isNull()
).show()

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+



In [18]:
#12
inventory_df.filter(
    inventory_df.stock_quantity.isNull()
).show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+



In [19]:
#13
sales_df.filter(
    sales_df.sale_amount.isNull()
).show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL|         UPI|
+-------+--------+----------+----------+-------------+-----------+------------+



In [20]:
#14
sales_df.filter(
    sales_df.payment_mode.isNull()
).show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1010|    S101|      P104|2026-01-16|            2|      14000|        NULL|
+-------+--------+----------+----------+-------------+-----------+------------+



In [21]:
#15
products_df = products_df.na.fill(
    {"supplier_id":"Unknown"}
)

products_df.show()


+----------+------------+-----------+------------+-----------+----------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|
+----------+------------+-----------+------------+-----------+----------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|
|      P103|  Television|Electronics|          LG|       S203|     45000|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|
|      P108|    Backpack|    Fashion|   Wildcraft|       S206|      2500|
|      P109|Refrigerator|Electronics|   Whirlpool|       S203|     38000|
|      P110|        Sofa|  Furniture|      Godrej|       S204|     32000|
|      P111|  Headphones|Electronics| 

In [22]:
#16
inventory_df = inventory_df.na.fill(
    {"stock_quantity":0}
)

inventory_df.show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|
|       I1007|    S103|      P106|            30|           10| 2026-01-14|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|
|       I1009|    S105|      P108|            50|           20| 2026-01-15|
|       I1010|    S106|      P109|             0|            6| 2026-01-16|
|       I101

In [23]:
#17
sales_df = sales_df.na.fill(
    {"sale_amount":0}
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|         UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|
| SA1010|    S101|      P104|2026-01-16|

In [24]:
#18
sales_df = sales_df.na.fill(
    {"payment_mode":"Unknown"}
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|         UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|
| SA1010|    S101|      P104|2026-01-16|

In [25]:
#19
from pyspark.sql.functions import when,col

sales_df = sales_df.withColumn(
    "data_quality_status",
    when(
        (col("sale_amount") == 0) |
        (col("payment_mode") == "Unknown"),
        "Incomplete"
    ).otherwise("Complete")
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|           Complete|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|           Complete|


In [26]:
#20
sales_df.groupBy(
    "data_quality_status"
).count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   13|
|         Incomplete|    2|
+-------------------+-----+



In [27]:
products_df.write.mode("overwrite").parquet("silver_products")

inventory_df.write.mode("overwrite").parquet("silver_inventory")

sales_df.write.mode("overwrite").parquet("silver_sales")

In [28]:
#21
suppliers_df.show(truncate=False)

+---------+---------------------------------+------+-----------+------------------------+
|city     |contact                          |rating|supplier_id|supplier_name           |
+---------+---------------------------------+------+-----------+------------------------+
|Hyderabad|{techsource@mail.com, 9876500011}|4.5   |S201       |TechSource India        |
|Bangalore|{mobileworld@mail.com, NULL}     |4.2   |S202       |MobileWorld Distributors|
|Mumbai   |{NULL, 9876500013}               |4.4   |S203       |HomeTech Supply         |
|Delhi    |{urban@mail.com, 9876500014}     |4.0   |S204       |Urban Furniture Co      |
|Pune     |{NULL, NULL}                     |3.8   |S205       |Fashion Direct          |
+---------+---------------------------------+------+-----------+------------------------+



In [29]:
#22
suppliers_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)



In [30]:
#23
from pyspark.sql.functions import col

suppliers_df.select(
    "supplier_id",
    "supplier_name",
    col("contact.phone").alias("phone")
).show()

+-----------+--------------------+----------+
|supplier_id|       supplier_name|     phone|
+-----------+--------------------+----------+
|       S201|    TechSource India|9876500011|
|       S202|MobileWorld Distr...|      NULL|
|       S203|     HomeTech Supply|9876500013|
|       S204|  Urban Furniture Co|9876500014|
|       S205|      Fashion Direct|      NULL|
+-----------+--------------------+----------+



In [31]:
#24
suppliers_df.select(
    "supplier_id",
    "supplier_name",
    col("contact.email").alias("email")
).show()

+-----------+--------------------+--------------------+
|supplier_id|       supplier_name|               email|
+-----------+--------------------+--------------------+
|       S201|    TechSource India| techsource@mail.com|
|       S202|MobileWorld Distr...|mobileworld@mail.com|
|       S203|     HomeTech Supply|                NULL|
|       S204|  Urban Furniture Co|      urban@mail.com|
|       S205|      Fashion Direct|                NULL|
+-----------+--------------------+--------------------+



In [32]:
#25
flat_suppliers_df = suppliers_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

flat_suppliers_df.show()

+-----------+--------------------+---------+------+----------+--------------------+
|supplier_id|       supplier_name|     city|rating|     phone|               email|
+-----------+--------------------+---------+------+----------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|      NULL|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|9876500013|                NULL|
|       S204|  Urban Furniture Co|    Delhi|   4.0|9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|      NULL|                NULL|
+-----------+--------------------+---------+------+----------+--------------------+



In [33]:
#26
flat_suppliers_df.filter(
    flat_suppliers_df.phone.isNull()
).show()

+-----------+--------------------+---------+------+-----+--------------------+
|supplier_id|       supplier_name|     city|rating|phone|               email|
+-----------+--------------------+---------+------+-----+--------------------+
|       S202|MobileWorld Distr...|Bangalore|   4.2| NULL|mobileworld@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8| NULL|                NULL|
+-----------+--------------------+---------+------+-----+--------------------+



In [34]:
#27
flat_suppliers_df.filter(
    flat_suppliers_df.email.isNull()
).show()

+-----------+---------------+------+------+----------+-----+
|supplier_id|  supplier_name|  city|rating|     phone|email|
+-----------+---------------+------+------+----------+-----+
|       S203|HomeTech Supply|Mumbai|   4.4|9876500013| NULL|
|       S205| Fashion Direct|  Pune|   3.8|      NULL| NULL|
+-----------+---------------+------+------+----------+-----+



In [35]:
#28
flat_suppliers_df = flat_suppliers_df.na.fill(
    {"phone":"Not Available"}
)

flat_suppliers_df.show()

+-----------+--------------------+---------+------+-------------+--------------------+
|supplier_id|       supplier_name|     city|rating|        phone|               email|
+-----------+--------------------+---------+------+-------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|   9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Available|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|   9876500013|                NULL|
|       S204|  Urban Furniture Co|    Delhi|   4.0|   9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|Not Available|                NULL|
+-----------+--------------------+---------+------+-------------+--------------------+



In [36]:
#29
flat_suppliers_df = flat_suppliers_df.na.fill(
    {"email":"Not Available"}
)

flat_suppliers_df.show()

+-----------+--------------------+---------+------+-------------+--------------------+
|supplier_id|       supplier_name|     city|rating|        phone|               email|
+-----------+--------------------+---------+------+-------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|   9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Available|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|   9876500013|       Not Available|
|       S204|  Urban Furniture Co|    Delhi|   4.0|   9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|Not Available|       Not Available|
+-----------+--------------------+---------+------+-------------+--------------------+



In [37]:
#30
flat_suppliers_df.agg(
    {"rating":"avg"}
).show()

+-----------------+
|      avg(rating)|
+-----------------+
|4.180000000000001|
+-----------------+



In [38]:
flat_suppliers_df.write.mode(
    "overwrite"
).parquet(
    "silver_suppliers"
)

In [39]:
#31
flat_suppliers_df.write.mode(
    "overwrite"
).parquet(
    "silver_suppliers"
)

In [40]:
#32
sales_products_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
)

sales_products_df.show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+
|      P101| SA1001|    S101|2026-01-10|            1|      65000|         UPI|           Complete|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P102| SA1002|    S101|2026-01-10|            2|      50000|        Card|           Complete|      Mobile|Electronics|     Samsung|       S202|     25000|
|      P101| SA1003|    S102|2026-01-11|            1|      65000|         UPI|           Complete|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P106| SA1004|    S103|2026-

In [41]:
#33
sales_stores_df = sales_df.join(
    stores_df,
    on="store_id",
    how="inner"
)

sales_stores_df.show()

+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|store_id|sale_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|          store_name|     city|      state| store_type|manager_name|
+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|    S101| SA1001|      P101|2026-01-10|            1|      65000|         UPI|           Complete|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S101| SA1002|      P102|2026-01-10|            2|      50000|        Card|           Complete|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102| SA1003|      P101|2026-01-11|            1|      65000|         UPI|           Complete|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya

In [42]:
#34
product_supplier_df = products_df.join(
    flat_suppliers_df,
    on="supplier_id",
    how="left"
)

product_supplier_df.show()

+-----------+----------+------------+-----------+------------+----------+--------------------+---------+------+-------------+--------------------+
|supplier_id|product_id|product_name|   category|       brand|unit_price|       supplier_name|     city|rating|        phone|               email|
+-----------+----------+------------+-----------+------------+----------+--------------------+---------+------+-------------+--------------------+
|       S201|      P101|      Laptop|Electronics|      Lenovo|     65000|    TechSource India|Hyderabad|   4.5|   9876500011| techsource@mail.com|
|       S202|      P102|      Mobile|Electronics|     Samsung|     25000|MobileWorld Distr...|Bangalore|   4.2|Not Available|mobileworld@mail.com|
|       S203|      P103|  Television|Electronics|          LG|     45000|     HomeTech Supply|   Mumbai|   4.4|   9876500013|       Not Available|
|       S204|      P104|Office Chair|  Furniture| Featherlite|      7000|  Urban Furniture Co|    Delhi|   4.0|   9876

In [43]:
#35
product_supplier_df.filter(
    product_supplier_df.supplier_name.isNull()
).show()

+-----------+----------+------------+-----------+---------+----------+-------------+----+------+-----+-----+
|supplier_id|product_id|product_name|   category|    brand|unit_price|supplier_name|city|rating|phone|email|
+-----------+----------+------------+-----------+---------+----------+-------------+----+------+-----+-----+
|       S206|      P107|       Watch|    Fashion| Fastrack|      8000|         NULL|NULL|  NULL| NULL| NULL|
|       S206|      P108|    Backpack|    Fashion|Wildcraft|      2500|         NULL|NULL|  NULL| NULL| NULL|
|       S999|      P111|  Headphones|Electronics|     Sony|      3000|         NULL|NULL|  NULL| NULL| NULL|
|    Unknown|      P112|     T-Shirt|    Fashion|     Puma|      1500|         NULL|NULL|  NULL| NULL| NULL|
+-----------+----------+------------+-----------+---------+----------+-------------+----+------+-----+-----+



In [44]:
#36
inventory_df.join(
    products_df,
    on="product_id",
    how="left"
).filter(
    products_df.product_name.isNull()
).show()

+----------+------------+--------+--------------+-------------+-----------+------------+--------+-----+-----------+----------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+--------------+-------------+-----------+------------+--------+-----+-----------+----------+
|      P120|       I1012|    S108|            12|            5| 2026-01-18|        NULL|    NULL| NULL|       NULL|      NULL|
+----------+------------+--------+--------------+-------------+-----------+------------+--------+-----+-----------+----------+



In [45]:
#37
sales_df.join(
    products_df,
    on="product_id",
    how="left"
).filter(
    products_df.product_name.isNull()
).show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+--------+-----+-----------+----------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|product_name|category|brand|supplier_id|unit_price|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+--------+-----+-----------+----------+
|      P120| SA1009|    S108|2026-01-15|            2|      10000|        Cash|           Complete|        NULL|    NULL| NULL|       NULL|      NULL|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+--------+-----+-----------+----------+



In [53]:
#38
from pyspark.sql.functions import col

# Rename 'city' column in stores_df to 'store_city' before joining
stores_df_renamed = stores_df.withColumnRenamed("city", "store_city")

retail_master_df = sales_df.join(
    stores_df_renamed,
    on="store_id",
    how="left"
).join(
    products_df,
    on="product_id",
    how="left"
)

retail_master_df.show()

+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+----------+-----------+-----------+------------+------------+-----------+------------+-----------+----------+
|product_id|store_id|sale_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|          store_name|store_city|      state| store_type|manager_name|product_name|   category|       brand|supplier_id|unit_price|
+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+----------+-----------+-----------+------------+------------+-----------+------------+-----------+----------+
|      P101|    S101| SA1001|2026-01-10|            1|      65000|         UPI|           Complete|Metro Mart Hyderabad| Hyderabad|  Telangana|Supermarket|Rahul Sharma|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P102|    S101| SA1002|2026-01-10|            2|      50000|        Card| 

In [54]:
#39
# Rename 'city' column in flat_suppliers_df to 'supplier_city' before joining
flat_suppliers_df_renamed = flat_suppliers_df.withColumnRenamed("city", "supplier_city")

retail_master_df = retail_master_df.join(
    flat_suppliers_df_renamed,
    on="supplier_id",
    how="left"
)

retail_master_df.show()

+-----------+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+----------+-----------+-----------+------------+------------+-----------+------------+----------+--------------------+-------------+------+-------------+--------------------+
|supplier_id|product_id|store_id|sale_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|          store_name|store_city|      state| store_type|manager_name|product_name|   category|       brand|unit_price|       supplier_name|supplier_city|rating|        phone|               email|
+-----------+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+----------+-----------+-----------+------------+------------+-----------+------------+----------+--------------------+-------------+------+-------------+--------------------+
|       S201|      P101|    S101| SA1001|2026-01-10|            1|      65000| 

In [48]:
#40
retail_master_df.count()

15

In [56]:
retail_master_df.write.mode(
    "overwrite"
).parquet(
    "silver_retail_master"
)

In [59]:
#41
from pyspark.sql.functions import when,col

inventory_products_df = inventory_df.join(
    products_df,
    on="product_id",
    how="inner"
)

inventory_products_df = inventory_products_df.withColumn(
    "stock_status",
    when(
        col("stock_quantity") <= col("reorder_level"),
        "Reorder Required"
    ).otherwise(
        "Sufficient Stock"
    )
)

inventory_products_df.show()

+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+----------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|product_name|   category|       brand|supplier_id|unit_price|    stock_status|
+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+----------------+
|      P101|       I1004|    S102|             8|            5| 2026-01-12|      Laptop|Electronics|      Lenovo|       S201|     65000|Sufficient Stock|
|      P101|       I1001|    S101|            10|            5| 2026-01-10|      Laptop|Electronics|      Lenovo|       S201|     65000|Sufficient Stock|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|      Mobile|Electronics|     Samsung|       S202|     25000|Sufficient Stock|
|      P103|       I1005|    S102|             5|            4| 2026-01-12| 

In [60]:
#42
inventory_products_df.filter(
    inventory_products_df.stock_status == "Reorder Required"
).show()

+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+----------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|product_name|   category|       brand|supplier_id|unit_price|    stock_status|
+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+----------------+
|      P104|       I1003|    S101|             3|            5| 2026-01-11|Office Chair|  Furniture| Featherlite|       S204|      7000|Reorder Required|
|      P105|       I1006|    S103|             2|            5| 2026-01-13| Study Table|  Furniture|Urban Ladder|       S204|     12000|Reorder Required|
|      P107|       I1008|    S104|             4|            5| 2026-01-15|       Watch|    Fashion|    Fastrack|       S206|      8000|Reorder Required|
|      P109|       I1010|    S106|             0|            6| 2026-01-16|R

In [61]:
#43
from pyspark.sql.functions import when,col

products_df = products_df.withColumn(
    "price_category",
    when(col("unit_price") >= 50000,"Premium")
    .when(col("unit_price") >= 10000,"Standard")
    .otherwise("Budget")
)

products_df.show()

+----------+------------+-----------+------------+-----------+----------+--------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|price_category|
+----------+------------+-----------+------------+-----------+----------+--------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|       Premium|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|      Standard|
|      P103|  Television|Electronics|          LG|       S203|     45000|      Standard|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|        Budget|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|      Standard|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|        Budget|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|        Budget|
|      P108|    Backpack|    Fashion|   Wildcraft|       S206|      2500|        Budget|
|      P109|Refrigera

In [62]:
#44
products_df.groupBy(
    "price_category"
).count().show()

+--------------+-----+
|price_category|count|
+--------------+-----+
|       Premium|    1|
|        Budget|    6|
|      Standard|    5|
+--------------+-----+



In [63]:
#45
sales_df = sales_df.withColumn(
    "revenue_category",
    when(col("sale_amount") >= 50000,"High Revenue")
    .when(col("sale_amount") >= 15000,"Medium Revenue")
    .otherwise("Low Revenue")
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     Low Revenue|
| SA1006|    S105|      P108|2026-01-13|            5|      1250

In [64]:
#46
sales_df.groupBy(
    "revenue_category"
).count().show()

+----------------+-----+
|revenue_category|count|
+----------------+-----+
|  Medium Revenue|    5|
|     Low Revenue|    7|
|    High Revenue|    3|
+----------------+-----+



In [65]:
#47
from pyspark.sql.functions import month

sales_df = sales_df.withColumn(
    "month",
    month("sale_date")
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     Low Revenue|    1|
| SA1006|    S10

In [66]:
#48
from pyspark.sql.functions import year

sales_df = sales_df.withColumn(
    "year",
    year("sale_date")
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     

In [67]:
#49
inventory_products_df = inventory_products_df.withColumn(
    "inventory_value",
    col("stock_quantity") * col("unit_price")
)

inventory_products_df.show()

+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+----------------+---------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|product_name|   category|       brand|supplier_id|unit_price|    stock_status|inventory_value|
+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+----------------+---------------+
|      P101|       I1004|    S102|             8|            5| 2026-01-12|      Laptop|Electronics|      Lenovo|       S201|     65000|Sufficient Stock|         520000|
|      P101|       I1001|    S101|            10|            5| 2026-01-10|      Laptop|Electronics|      Lenovo|       S201|     65000|Sufficient Stock|         650000|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|      Mobile|Electronics|     Samsung|       S202|     25000|Sufficient Sto

In [68]:
#50
flat_suppliers_df = flat_suppliers_df.withColumn(
    "supplier_quality",
    when(col("rating") >= 4.5,"Excellent")
    .when(col("rating") >= 4.0,"Good")
    .otherwise("Average")
)

flat_suppliers_df.show()

+-----------+--------------------+---------+------+-------------+--------------------+----------------+
|supplier_id|       supplier_name|     city|rating|        phone|               email|supplier_quality|
+-----------+--------------------+---------+------+-------------+--------------------+----------------+
|       S201|    TechSource India|Hyderabad|   4.5|   9876500011| techsource@mail.com|       Excellent|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Available|mobileworld@mail.com|            Good|
|       S203|     HomeTech Supply|   Mumbai|   4.4|   9876500013|       Not Available|            Good|
|       S204|  Urban Furniture Co|    Delhi|   4.0|   9876500014|      urban@mail.com|            Good|
|       S205|      Fashion Direct|     Pune|   3.8|Not Available|       Not Available|         Average|
+-----------+--------------------+---------+------+-------------+--------------------+----------------+



In [69]:
#51
inventory_products_df.groupBy(
    "store_id"
).sum(
    "inventory_value"
).show()

+--------+--------------------+
|store_id|sum(inventory_value)|
+--------+--------------------+
|    S105|              125000|
|    S102|              745000|
|    S106|                   0|
|    S104|               32000|
|    S107|               32000|
|    S101|             1296000|
|    S103|              159000|
+--------+--------------------+



In [70]:
#52
inventory_products_df.groupBy(
    "category"
).sum(
    "inventory_value"
).show()

+-----------+--------------------+
|   category|sum(inventory_value)|
+-----------+--------------------+
|    Fashion|              292000|
|Electronics|             2020000|
|  Furniture|               77000|
+-----------+--------------------+



In [71]:
#53
inventory_products_df.filter(
    inventory_products_df.stock_status == "Reorder Required"
).count()

5

In [72]:
#54
from pyspark.sql.functions import sum

sales_df.agg(
    sum("sale_amount").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|       373000|
+-------------+



In [73]:
#55
retail_master_df.groupBy(
    "store_id",
    "store_name"
).sum(
    "sale_amount"
).show()

+--------+--------------------+----------------+
|store_id|          store_name|sum(sale_amount)|
+--------+--------------------+----------------+
|    S105|    Metro Mart Delhi|           20000|
|    S102|Metro Mart Bangalore|           65000|
|    S101|Metro Mart Hyderabad|          154000|
|    S104|  Metro Mart Chennai|           24000|
|    S103|   Metro Mart Mumbai|           30000|
|    S108|   Metro Mart Jaipur|           10000|
|    S107|    Metro Mart Kochi|           32000|
|    S106|     Metro Mart Pune|           38000|
+--------+--------------------+----------------+



In [79]:
#56
retail_master_df.groupBy(
    "store_city"
).sum(
    "sale_amount"
).show()

+----------+----------------+
|store_city|sum(sale_amount)|
+----------+----------------+
| Bangalore|           65000|
|     Kochi|           32000|
|   Chennai|           24000|
|    Mumbai|           30000|
|      Pune|           38000|
|     Delhi|           20000|
| Hyderabad|          154000|
|    Jaipur|           10000|
+----------+----------------+



In [76]:
#57
retail_master_df.groupBy(
    "product_id",
    "product_name"
).sum(
    "sale_amount"
).show()


+----------+------------+----------------+
|product_id|product_name|sum(sale_amount)|
+----------+------------+----------------+
|      P107|       Watch|           24000|
|      P102|      Mobile|           75000|
|      P108|    Backpack|           20000|
|      P109|Refrigerator|           38000|
|      P110|        Sofa|           32000|
|      P120|        NULL|           10000|
|      P101|      Laptop|          130000|
|      P104|Office Chair|           14000|
|      P103|  Television|               0|
|      P106|       Shoes|           18000|
|      P105| Study Table|           12000|
+----------+------------+----------------+



In [77]:
#58
retail_master_df.groupBy(
    "product_id",
    "product_name"
).sum(
    "sale_amount"
).show()

+----------+------------+----------------+
|product_id|product_name|sum(sale_amount)|
+----------+------------+----------------+
|      P107|       Watch|           24000|
|      P102|      Mobile|           75000|
|      P108|    Backpack|           20000|
|      P109|Refrigerator|           38000|
|      P110|        Sofa|           32000|
|      P120|        NULL|           10000|
|      P101|      Laptop|          130000|
|      P104|Office Chair|           14000|
|      P103|  Television|               0|
|      P106|       Shoes|           18000|
|      P105| Study Table|           12000|
+----------+------------+----------------+



In [78]:
#59
retail_master_df.groupBy(
    "payment_mode"
).sum(
    "sale_amount"
).show()

+------------+----------------+
|payment_mode|sum(sale_amount)|
+------------+----------------+
|     Unknown|           14000|
|        Card|          133000|
|        Cash|           35500|
|         UPI|          190500|
+------------+----------------+



In [80]:
#60
retail_master_df.groupBy(
    "product_id",
    "product_name"
).sum(
    "sale_amount"
).orderBy(
    "sum(sale_amount)",
    ascending=False
).show(1)

+----------+------------+----------------+
|product_id|product_name|sum(sale_amount)|
+----------+------------+----------------+
|      P101|      Laptop|          130000|
+----------+------------+----------------+
only showing top 1 row


In [82]:
#61
from pyspark.sql.window import Window
from pyspark.sql.functions import rank,sum

product_revenue_df = retail_master_df.groupBy(
    "product_id",
    "product_name"
).agg(
    sum("sale_amount").alias("total_revenue")
)

window_spec = Window.orderBy(
    product_revenue_df.total_revenue.desc()
)

product_revenue_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+----------+------------+-------------+----+
|product_id|product_name|total_revenue|rank|
+----------+------------+-------------+----+
|      P101|      Laptop|       130000|   1|
|      P102|      Mobile|        75000|   2|
|      P109|Refrigerator|        38000|   3|
|      P110|        Sofa|        32000|   4|
|      P107|       Watch|        24000|   5|
|      P108|    Backpack|        20000|   6|
|      P106|       Shoes|        18000|   7|
|      P104|Office Chair|        14000|   8|
|      P105| Study Table|        12000|   9|
|      P120|        NULL|        10000|  10|
|      P103|  Television|            0|  11|
+----------+------------+-------------+----+



In [83]:
#62
store_revenue_df = retail_master_df.groupBy(
    "store_id",
    "store_name"
).agg(
    sum("sale_amount").alias("total_revenue")
)

window_spec = Window.orderBy(
    store_revenue_df.total_revenue.desc()
)

store_revenue_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+--------+--------------------+-------------+----+
|store_id|          store_name|total_revenue|rank|
+--------+--------------------+-------------+----+
|    S101|Metro Mart Hyderabad|       154000|   1|
|    S102|Metro Mart Bangalore|        65000|   2|
|    S106|     Metro Mart Pune|        38000|   3|
|    S107|    Metro Mart Kochi|        32000|   4|
|    S103|   Metro Mart Mumbai|        30000|   5|
|    S104|  Metro Mart Chennai|        24000|   6|
|    S105|    Metro Mart Delhi|        20000|   7|
|    S108|   Metro Mart Jaipur|        10000|   8|
+--------+--------------------+-------------+----+



In [84]:
#63
category_revenue_df = retail_master_df.groupBy(
    "category",
    "product_id",
    "product_name"
).agg(
    sum("sale_amount").alias("total_revenue")
)

category_window = Window.partitionBy(
    "category"
).orderBy(
    category_revenue_df.total_revenue.desc()
)

category_revenue_df.withColumn(
    "rank",
    rank().over(category_window)
).show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|Electronics|      P102|      Mobile|        75000|   2|
|Electronics|      P109|Refrigerator|        38000|   3|
|Electronics|      P103|  Television|            0|   4|
|    Fashion|      P107|       Watch|        24000|   1|
|    Fashion|      P108|    Backpack|        20000|   2|
|    Fashion|      P106|       Shoes|        18000|   3|
|  Furniture|      P110|        Sofa|        32000|   1|
|  Furniture|      P104|Office Chair|        14000|   2|
|  Furniture|      P105| Study Table|        12000|   3|
+-----------+----------+------------+-------------+----+



In [85]:
#64
category_revenue_df.withColumn(
    "rank",
    rank().over(category_window)
).filter(
    "rank = 1"
).show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|    Fashion|      P107|       Watch|        24000|   1|
|  Furniture|      P110|        Sofa|        32000|   1|
+-----------+----------+------------+-------------+----+



In [86]:
#65
category_revenue_df.withColumn(
    "rank",
    rank().over(category_window)
).filter(
    "rank <= 3"
).show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|Electronics|      P102|      Mobile|        75000|   2|
|Electronics|      P109|Refrigerator|        38000|   3|
|    Fashion|      P107|       Watch|        24000|   1|
|    Fashion|      P108|    Backpack|        20000|   2|
|    Fashion|      P106|       Shoes|        18000|   3|
|  Furniture|      P110|        Sofa|        32000|   1|
|  Furniture|      P104|Office Chair|        14000|   2|
|  Furniture|      P105| Study Table|        12000|   3|
+-----------+----------+------------+-------------+----+



In [87]:
#66
store_state_df = retail_master_df.groupBy(
    "state",
    "store_id",
    "store_name"
).agg(
    sum("sale_amount").alias("total_revenue")
)

state_window = Window.partitionBy(
    "state"
).orderBy(
    store_state_df.total_revenue.desc()
)

store_state_df.withColumn(
    "rank",
    rank().over(state_window)
).filter(
    "rank = 1"
).show()

+-----------+--------+--------------------+-------------+----+
|      state|store_id|          store_name|total_revenue|rank|
+-----------+--------+--------------------+-------------+----+
|      Delhi|    S105|    Metro Mart Delhi|        20000|   1|
|  Karnataka|    S102|Metro Mart Bangalore|        65000|   1|
|     Kerala|    S107|    Metro Mart Kochi|        32000|   1|
|Maharashtra|    S106|     Metro Mart Pune|        38000|   1|
|  Rajasthan|    S108|   Metro Mart Jaipur|        10000|   1|
| Tamil Nadu|    S104|  Metro Mart Chennai|        24000|   1|
|  Telangana|    S101|Metro Mart Hyderabad|       154000|   1|
+-----------+--------+--------------------+-------------+----+



In [88]:
#67
from pyspark.sql.functions import sum

running_window = Window.orderBy("sale_date")

sales_df.withColumn(
    "running_total",
    sum("sale_amount").over(running_window)
).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|running_total|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|       115000|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|       115000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|       180000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|       206000|
| SA10

In [89]:
#68
from pyspark.sql.functions import lag

sales_df.withColumn(
    "previous_sale",
    lag("sale_amount").over(
        Window.orderBy("sale_date")
    )
).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|previous_sale|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|         NULL|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|        65000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|        50000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|        65000|
| SA10

In [90]:
#69
from pyspark.sql.functions import lead

sales_df.withColumn(
    "next_sale",
    lead("sale_amount").over(
        Window.orderBy("sale_date")
    )
).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|next_sale|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+---------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|    50000|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|    65000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|    18000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|     8000|
| SA1005|    S104|      P107|2026-

In [91]:
#70
from pyspark.sql.functions import lag,col

sales_compare = sales_df.withColumn(
    "previous_sale",
    lag("sale_amount").over(
        Window.orderBy("sale_date")
    )
)

sales_compare.filter(
    col("sale_amount") > col("previous_sale")
).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|previous_sale|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|        50000|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|           Complete|     Low Revenue|    1|2026|         8000|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|           Complete|  Medium Revenue|    1|2026|        12500|
| SA1010|    S101|      P104|2026-01-16|            2|      14000|     Unknown|         Incomplete|     Low Revenue|    1|2026|        10000|
| SA10

In [92]:
#71
stores_df.createOrReplaceTempView("stores")

products_df.createOrReplaceTempView("products")

inventory_df.createOrReplaceTempView("inventory")

sales_df.createOrReplaceTempView("sales")

flat_suppliers_df.createOrReplaceTempView("suppliers")

In [93]:
#72
spark.sql("""
SELECT *
FROM sales
""").show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     

In [94]:
#73
spark.sql("""
SELECT category,
       COUNT(*) AS product_count
FROM products
GROUP BY category
""").show()

+-----------+-------------+
|   category|product_count|
+-----------+-------------+
|    Fashion|            4|
|Electronics|            5|
|  Furniture|            3|
+-----------+-------------+



In [95]:
#74
spark.sql("""
SELECT store_id,
       SUM(sale_amount) AS revenue
FROM sales
GROUP BY store_id
ORDER BY revenue DESC
""").show()

+--------+-------+
|store_id|revenue|
+--------+-------+
|    S101| 154000|
|    S102|  65000|
|    S106|  38000|
|    S107|  32000|
|    S103|  30000|
|    S104|  24000|
|    S105|  20000|
|    S108|  10000|
+--------+-------+



In [97]:
#75
sales_city_df = retail_master_df.select(
    "store_city",
    "sale_amount"
)

sales_city_df.createOrReplaceTempView(
    "sales_city"
)

spark.sql("""
SELECT store_city,
       SUM(sale_amount) AS revenue
FROM sales_city
GROUP BY store_city
ORDER BY revenue DESC
""").show()

+----------+-------+
|store_city|revenue|
+----------+-------+
| Hyderabad| 154000|
| Bangalore|  65000|
|      Pune|  38000|
|     Kochi|  32000|
|    Mumbai|  30000|
|   Chennai|  24000|
|     Delhi|  20000|
|    Jaipur|  10000|
+----------+-------+



In [98]:
#76
retail_master_df.write.mode(
    "overwrite"
).parquet(
    "gold_sales"
)

In [101]:
#77
from pyspark.sql.functions import year, month

retail_master_df = retail_master_df.withColumn(
    "year",
    year("sale_date")
).withColumn(
    "month",
    month("sale_date")
)

retail_master_df.write.mode(
    "overwrite"
).partitionBy(
    "year",
    "month"
).parquet(
    "gold_sales_partitioned"
)

In [100]:
#78
march_sales = [
("SA1016","S101","P101","2026-03-01",1,65000,"UPI"),
("SA1017","S102","P103","2026-03-02",1,45000,"Card"),
("SA1018","S103","P106","2026-03-03",2,9000,"Cash")
]

incremental_sales_df = spark.createDataFrame(
    march_sales,
    [
        "sale_id",
        "store_id",
        "product_id",
        "sale_date",
        "quantity_sold",
        "sale_amount",
        "payment_mode"
    ]
)

incremental_sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1016|    S101|      P101|2026-03-01|            1|      65000|         UPI|
| SA1017|    S102|      P103|2026-03-02|            1|      45000|        Card|
| SA1018|    S103|      P106|2026-03-03|            2|       9000|        Cash|
+-------+--------+----------+----------+-------------+-----------+------------+



In [102]:
#79
incremental_sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1016|    S101|      P101|2026-03-01|            1|      65000|         UPI|
| SA1017|    S102|      P103|2026-03-02|            1|      45000|        Card|
| SA1018|    S103|      P106|2026-03-03|            2|       9000|        Cash|
+-------+--------+----------+----------+-------------+-----------+------------+



In [103]:
#80
incremental_sales_df.write.mode(
    "append"
).parquet(
    "silver_sales"
)

In [106]:
#81
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, LongType

# Define the schema to explicitly handle the type promotion for sale_amount and quantity_sold
schema = StructType([
    StructField("sale_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("sale_date", DateType(), True),
    StructField("quantity_sold", LongType(), True), # Promote to LongType
    StructField("sale_amount", LongType(), True),   # Promote to LongType
    StructField("payment_mode", StringType(), True),
    StructField("data_quality_status", StringType(), True),
    StructField("revenue_category", StringType(), True),
    StructField("month", IntegerType(), True),
    StructField("year", IntegerType(), True)
])

updated_sales_df = spark.read.schema(schema).parquet(
    "silver_sales"
)

updated_sales_df.groupBy(
    "product_id"
).sum(
    "sale_amount"
).show()

+----------+----------------+
|product_id|sum(sale_amount)|
+----------+----------------+
|      P110|           32000|
|      P105|           12000|
|      P102|           75000|
|      P106|           27000|
|      P107|           24000|
|      P120|           10000|
|      P103|           45000|
|      P109|           38000|
|      P104|           14000|
|      P101|          195000|
|      P108|           20000|
+----------+----------------+



In [107]:
#82
updated_sales_df.groupBy(
    "store_id"
).sum(
    "sale_amount"
).show()

+--------+----------------+
|store_id|sum(sale_amount)|
+--------+----------------+
|    S105|           20000|
|    S102|          110000|
|    S106|           38000|
|    S104|           24000|
|    S107|           32000|
|    S101|          219000|
|    S108|           10000|
|    S103|           39000|
+--------+----------------+



In [108]:
#83
from pyspark.sql.functions import year,month

updated_sales_df = updated_sales_df.withColumn(
    "year",
    year("sale_date")
)

updated_sales_df = updated_sales_df.withColumn(
    "month",
    month("sale_date")
)

updated_sales_df.write.mode(
    "overwrite"
).partitionBy(
    "year",
    "month"
).parquet(
    "gold_sales_v2"
)

Py4JJavaError: An error occurred while calling o1494.parquet.
: org.apache.spark.SparkException: [FAILED_READ_FILE.PARQUET_COLUMN_DATA_TYPE_MISMATCH] Encountered error while reading file file:///content/silver_sales/part-00001-b9337afc-891a-4493-8b0e-e3c05697b754-c000.snappy.parquet. Data type mismatches when reading Parquet column [sale_date]. Expected Spark type date, actual Parquet type BINARY. SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.parquetColumnDataTypeMismatchError(QueryExecutionErrors.scala:847)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:138)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:695)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.

In [109]:
#84
print(
    "Original Sales Count:",
    sales_df.count()
)

print(
    "Updated Sales Count:",
    updated_sales_df.count()
)

Original Sales Count: 15
Updated Sales Count: 18


In [116]:
#85
retail_master_df.groupBy(
    "store_id",
    "store_name",
    "store_city",
    "state"
).agg(
    sum("quantity_sold").alias("total_sales"),
    sum("sale_amount").alias("total_revenue")
).show()

+--------+--------------------+----------+-----------+-----------+-------------+
|store_id|          store_name|store_city|      state|total_sales|total_revenue|
+--------+--------------------+----------+-----------+-----------+-------------+
|    S106|     Metro Mart Pune|      Pune|Maharashtra|          1|        38000|
|    S107|    Metro Mart Kochi|     Kochi|     Kerala|          1|        32000|
|    S101|Metro Mart Hyderabad| Hyderabad|  Telangana|          6|       154000|
|    S103|   Metro Mart Mumbai|    Mumbai|Maharashtra|          5|        30000|
|    S105|    Metro Mart Delhi|     Delhi|      Delhi|          8|        20000|
|    S102|Metro Mart Bangalore| Bangalore|  Karnataka|          2|        65000|
|    S104|  Metro Mart Chennai|   Chennai| Tamil Nadu|          3|        24000|
|    S108|   Metro Mart Jaipur|    Jaipur|  Rajasthan|          2|        10000|
+--------+--------------------+----------+-----------+-----------+-------------+



In [111]:
#86
retail_master_df.groupBy(
    "product_id",
    "product_name",
    "category",
    "brand"
).agg(
    sum("quantity_sold").alias("total_quantity_sold"),
    sum("sale_amount").alias("total_revenue")
).show()

+----------+------------+-----------+------------+-------------------+-------------+
|product_id|product_name|   category|       brand|total_quantity_sold|total_revenue|
+----------+------------+-----------+------------+-------------------+-------------+
|      P110|        Sofa|  Furniture|      Godrej|                  1|        32000|
|      P104|Office Chair|  Furniture| Featherlite|                  2|        14000|
|      P101|      Laptop|Electronics|      Lenovo|                  2|       130000|
|      P109|Refrigerator|Electronics|   Whirlpool|                  1|        38000|
|      P105| Study Table|  Furniture|Urban Ladder|                  1|        12000|
|      P107|       Watch|    Fashion|    Fastrack|                  3|        24000|
|      P102|      Mobile|Electronics|     Samsung|                  3|        75000|
|      P108|    Backpack|    Fashion|   Wildcraft|                  8|        20000|
|      P106|       Shoes|    Fashion|        Nike|               

In [112]:
#87
inventory_products_df.select(
    "store_id",
    "product_id",
    "product_name",
    "stock_quantity",
    "reorder_level",
    "stock_status"
).show()

+--------+----------+------------+--------------+-------------+----------------+
|store_id|product_id|product_name|stock_quantity|reorder_level|    stock_status|
+--------+----------+------------+--------------+-------------+----------------+
|    S101|      P101|      Laptop|            10|            5|Sufficient Stock|
|    S101|      P102|      Mobile|            25|           10|Sufficient Stock|
|    S101|      P104|Office Chair|             3|            5|Reorder Required|
|    S102|      P101|      Laptop|             8|            5|Sufficient Stock|
|    S102|      P103|  Television|             5|            4|Sufficient Stock|
|    S103|      P105| Study Table|             2|            5|Reorder Required|
|    S103|      P106|       Shoes|            30|           10|Sufficient Stock|
|    S104|      P107|       Watch|             4|            5|Reorder Required|
|    S105|      P108|    Backpack|            50|           20|Sufficient Stock|
|    S106|      P109|Refrige

In [113]:
#88
flat_suppliers_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    "supplier_quality",
    "phone",
    "email"
).show()

+-----------+--------------------+---------+------+----------------+-------------+--------------------+
|supplier_id|       supplier_name|     city|rating|supplier_quality|        phone|               email|
+-----------+--------------------+---------+------+----------------+-------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|       Excellent|   9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|            Good|Not Available|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|            Good|   9876500013|       Not Available|
|       S204|  Urban Furniture Co|    Delhi|   4.0|            Good|   9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|         Average|Not Available|       Not Available|
+-----------+--------------------+---------+------+----------------+-------------+--------------------+



In [114]:
#89
retail_master_df.groupBy(
    "category"
).agg(
    sum("quantity_sold").alias("total_quantity_sold"),
    sum("sale_amount").alias("total_revenue")
).show()

+-----------+-------------------+-------------+
|   category|total_quantity_sold|total_revenue|
+-----------+-------------------+-------------+
|    Fashion|                 15|        62000|
|       NULL|                  2|        10000|
|Electronics|                  7|       243000|
|  Furniture|                  4|        58000|
+-----------+-------------------+-------------+



In [115]:
#90
retail_master_df.groupBy(
    "payment_mode"
).agg(
    sum("sale_amount").alias("total_revenue")
).show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|     Unknown|        14000|
|        Card|       133000|
|        Cash|        35500|
|         UPI|       190500|
+------------+-------------+

